# 第26讲：tokenizer_gpt4o

按原单元编号阅读。配套讲解见本讲的解释.md，运行方法与记录见复现目录。


### 原单元 0（markdown）

原课微信讨论链接保留，当前访问失败，未核实文章正文；模型训练细节按官方资料阅读。


- https://mp.weixin.qq.com/s/BnwGitYV59LXqaenP3dR6w

### 原单元 1（markdown）

这里比较两套编码。每套编码的ID语义需要与使用它的模型权重匹配。


## `cl100k_base` => `o200k_base`

### 原单元 2（code）

tiktoken提供真实字节BPE，langdetect是统计语言识别，regex支持Unicode属性和更丰富正则。import不加载GPT-4o权重，不调用API。


In [15]:
import tiktoken
import langdetect
import regex as re

### 原单元 3（code）

get_encoding按编码名读取注册构造器，首次可能下载词表并校验SHA，返回Encoding对象，后续缓存复用。本次预置真实完整词表缓存，原调用不改且禁止联网验证。


In [2]:
# https://platform.openai.com/tokenizer
T1 = tiktoken.get_encoding('cl100k_base')
T2 = tiktoken.get_encoding('o200k_base')

### 原单元 4（code）

cl100k把此英文句子转5个ID。ID是词表行号，不是Unicode码点；没有embedding/模型前向。


In [3]:
T1.encode('i love chatgpt')

[72, 3021, 6369, 70, 418]

### 原单元 5（code）

o200k同句仍是5个ID但数值不同，直接反例说明词表更大不保证每句token严格更少。


In [4]:
T2.encode('i love chatgpt')

[72, 3047, 7999, 70, 555]

### 原单元 6（code）

n_vocab=max_token_value+1，分别100277与200019，不等于所有中间ID都有效。cl普通100256+特殊5共100261，o普通199998+特殊2共200000，其余是洞位。


In [5]:
T1.n_vocab, T2.n_vocab

(100277, 200019)

### 原单元 7（markdown）

原课讨论压缩token_ids，是相同原文使用更少token，不是把每个整数ID本身压成更少bits。


## 更大的词表 => 压缩 token_ids

### 原单元 8（markdown）

原文更大词表必更少和价格更低需加条件：两词表/预切分不同，非集合嵌套；总账单还涉及模型单价、输入/输出/缓存等。这里只测文本长度，不引用当前价格。


- https://openai.com/index/hello-gpt-4o/
- 更大的词表，一段话（sentence）encoder 之后的 token_id 会变少，也就是一些词（甚至词组）会更容易命中词表，而不再会被拆分为多个 token_ids
    - 从经济上的考虑，api price 就会更低

### 原单元 9（code）

cl对指定中文完整句编码后len=34，len数列表元素，不是字数、字节数或张量长度。


In [20]:
len(T1.encode('你好，我的名字是GPT-4o。我是一种新型的语言模型，很高兴见到你!'))

34

### 原单元 10（code）

o对同一句len=24，减少10个即29.41%，旧/新=1.4167；不同文本不保证同比例。


In [21]:
len(T2.encode('你好，我的名字是GPT-4o。我是一种新型的语言模型，很高兴见到你!'))

24

### 原单元 11（code）

with确保文件关闭，encoding指定UTF8；readlines一次读入全部文本再逐行编码累加，不是流式训练。相对路径./xyj.txt配套原文件完整发布，保留原字节/原换行；Python文本模式会规范化换行。


In [6]:
token_ids_len1 = 0
token_ids_len2 = 0
with open('./xyj.txt', 'r', encoding='utf-8') as f:
    for line in f.readlines():
        token_ids_len1 += len(T1.encode(line))
        token_ids_len2 += len(T2.encode(line))

### 原单元 12（code）

两个整数及比率：cl990508/o703303，旧/新1.4083659532，新/旧0.7100427256，减少约29%。这是逐行全文统计，不是任意中文或整篇一次编码的定理。


In [7]:
token_ids_len1, token_ids_len2, token_ids_len1/token_ids_len2, token_ids_len2/token_ids_len1

(990508, 703303, 1.408365953223575, 0.7100427255509294)

### 原单元 13（markdown）

进入中文词表观察：看token对应文本，不能直接测embedding训练充分程度或GPT回答能力。


## gpt-4o 中文 tokenizer

### 原单元 14（markdown）

原文关于语料过滤导致异常回答的因果解释是推测，notebook未提供GPT-4o训练日志、语料、权重或受控实验。分享链接历史回答不能由本次分词实验重现；gist是第三方观察脚本，不是官方训练说明。


- 先看一个现象：https://chat.openai.com/share/f4227bf3-bc46-43d1-a982-f7dea78702ed
    - “给主人留下些什么吧”这句话翻译成英文
- 出于压缩tokens的目的，先在最广泛的基础语料里，训练了一把tokenizer，但没做过多的过滤，而在实际训练 transformer 的过程中，又过滤了基础语料的低质量数据集，导致那些只在基础语料中大量出现的token 未被充分训练；
- https://gist.github.com/ctlllll/4451e94f3b2ca415515f3ee369c8c374

### 原单元 15（code）

原词组在o200k编码为单个177431。它属于普通mergeable token，不是special_tokens里的保留控制符；原课特殊中文tokens是口语描述。


In [22]:
T2.encode("给主人留下些什么吧")

[177431]

### 原单元 16（code）

同词组在cl100k拆成12个ID；比较必须连同完整解码检查，单token可能只是UTF8字符的一部分。


In [23]:
T1.encode("给主人留下些什么吧")

[90112, 36668, 17792, 40198, 247, 17297, 98184, 6271, 222, 82696, 7305, 100]

### 原单元 17（code）

扫描o200k所有range(n_vocab)位置，包括19个无效洞位；decode默认替换坏UTF8，len数Python字符。bare except吞掉Rust PanicException甚至中断；按长度降序稳定排序后langdetect筛zh-cn，实际只打印20条，不是注释写的100。语言检测对短串不可靠且有随机性，本次外置seed26；保留原算法/历史panic输出。


In [8]:
length_dict = {}

for i in range(T2.n_vocab):
    try:
        length_dict[i] = len(T2.decode([i]))
    except:
        pass
      
# Sort by length
length_dict = dict(sorted(length_dict.items(), key=lambda item: -item[1]))
# print(length_dict)
# Print the top 100 chinese wordsÅ
tot = 0
for item in length_dict:
    try:
        if langdetect.detect(T2.decode([item])) == "zh-cn":
            print(item, T2.decode([item]))
            tot += 1
    except:
        pass
    if tot == 20:
        break

thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:

185118 _日本毛片免费视频观看
116852  中国福利彩票天天
128031 久久免费热在线精品
154809 无码不卡高清免费v
172750  大发快三大小单双
177431 给主人留下些什么吧
181679  qq的天天中彩票
184969 _日本一级特黄大片
187822  大发快三开奖结果
49649  彩神争霸邀请码
89409 免费视频在线观看
122333 无码不卡高清免费
122712 无码一区二区三区
128600  大发时时彩计划
133274 】【：】【“】【
135161  大发时时彩开奖
149168  大发时时彩怎么
160029  大发快三是国家
160131  大发快三是不是
176039 精品一区二区三区


### 原单元 18（code）

同流程扫描cl100k，16个无效洞位。标点/换行也可能被langdetect判zh-cn，不能把输出理解成真实中文词频榜或训练语料分布。实际执行完整100277位置。


In [9]:
length_dict = {}

for i in range(T1.n_vocab):
    try:
        length_dict[i] = len(T1.decode([i]))
    except:
        pass
      
# Sort by length
length_dict = dict(sorted(length_dict.items(), key=lambda item: -item[1]))
# print(length_dict)
# Print the top 100 chinese words
tot = 0
for item in length_dict:
    try:
        if langdetect.detect(T1.decode([item])) == "zh-cn":
            print(item, T1.decode([item]))
            tot += 1
    except:
        pass
    if tot == 20:
        break

thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
n

75787 。






19066 。




54689 。。


63093 。',

64376 。",

65053 监听页面
76718 微软雅黑
80000 。”


92099 删除成功
97432 ”。


3490 。


28918 、


28966 ）


38232 。</
38365 。

41771  获取
47436 ：</
48858  创建
49055 请输入
49543 ：




### 原单元 19（code）

再次扫描完整o200k，以正则剔除全空白/非词字符/特定符号串；不是中文检测。英文、数字、代码片段仍可通过。tot==20退出，只取20条；bare except问题不改原版，在独立诊断提供安全写法。


In [19]:
length_dict = {}

for i in range(T2.n_vocab):
    try:
        length_dict[i] = len(T2.decode([i]))
    except:
        pass
      
# Sort by length
length_dict = dict(sorted(length_dict.items(), key=lambda item: -item[1]))
# print(length_dict)
# Print the top 100 chinese words
tot = 0
pattern = r'^[\s\W_=+\\-]*$'
for item in length_dict:
    try:
        # print(T2.decode([item]), re.match(pattern, T2.decode([item])))
        if not re.match(pattern, T2.decode([item])):
            print(item, T2.decode([item]))
            tot += 1
    except:
        pass
    if tot == 20:
        break

161518 abcdefghijklmnopqrstuvwxyz
184150 ABCDEFGHIJKLMNOPQRSTUVWXYZ
130756  verantwoordelijkheid
150141  สำนักเลขานุการองค์กร
106123  telecommunications
133739  selbstverständlich
135127  วิเคราะห์บอลวันนี้
154976 .onreadystatechange
166459  significativamente
184611  Telecommunications
193348  Wahrscheinlichkeit
197767  disproportionately
88004  unterschiedlichen
100106  interdisciplinary
117361 .githubusercontent
132622  responsabilidades
134381  Herausforderungen
135128  multidisciplinary
138955  STDMETHODCALLTYPE
198090  commercialization


thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
no entry found for key
thread '<unnamed>' panicked at src/lib.rs:179:64:
n

### 原单元 20（code）

原空代码格，保留；只新增说明注释，不补造训练/调用结果。
